In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
import tidy3d.web as web
import json
from typing import Optional

from autograd import value_and_grad
import autograd.numpy as anp

from copy import deepcopy
import optax

import os
import pickle

In [ ]:
wl_c = 1.55  # Central wavelength.
wl_bw = 0.100  # Wavelength bandwidth.
wl_n = 101  # Number of wavelength points to compute the output data.

n_sin = 2.0
n_sio2 = 1.444
n_si = 3.48
NA = 0.41
n_core = np.sqrt(NA**2 + n_sio2**2)

with open("n_core.json", "r") as f:
    data = json.load(f)
    n_core_test = data["n_core"] # Refractive index of core with which the mode radius is 3.2um, assuming SiO2 cladding.

mat_sin = td.Medium(permittivity=n_sin**2)  # Taper and substrate material.
mat_sio2 = td.Medium(permittivity=n_sio2**2)  # Box and cladding material.
mat_air = td.Medium(permittivity=1.00)  # External medium material.
mat_si = td.Medium(permittivity=n_si**2)  # Waveguide material.
mat_core = td.Medium(permittivity=n_core**2)
mat_core_test = td.Medium(permittivity=n_core_test**2)

In [ ]:
# Taper geometries and limits
taper_l = 280.0  # Taper length.
taper_l_min = 250.0  # Taper length.
taper_l_max = 330.0 # Taper length.

taper_w_in = 0.5  # Taper tip width.
taper_w_in_min = 0.1  # Taper tip width minimum.
taper_w_in_max = 0.8  # Taper tip width maximum.
taper_w_out = 1.2  # Taper output width.
taper_t = 0.22  # Taper thickness.

# Spot size of the gaussian mode launched by the lensed fiber at the taper tip.
spot_size = 3.2
core_diameter = 2.4
fiber_length = 8

# Fiber-PIC gap
gap = 0.01*wl_c

box_thick = 4.0  # Silicon dioxide box layer.
box_thick_min = 3.8
box_thick_max = 4.2
clad_thick = 4.0  # Silicon dioxide layer covering the taper.
clad_thick_min = 3.8
clad_thick_max = 4.2

In [ ]:
# Extra space around the taper at x,y directions.
pad_x = 1 * wl_c
pad_y = 1.5 * wl_c

# Wavelength and frequency values.
wl_range = np.linspace(wl_c - wl_bw / 2, wl_c + wl_bw / 2, wl_n)
freq_c = td.C_0 / wl_c
freq_range = td.C_0 / wl_range
freq_width = 0.5 * (np.max(freq_range) - np.min(freq_range))
run_time = 30 / freq_width

# Large number to be used in replacement of td.inf when necessary.
_inf = 1e3

# 1. Simulation setup

In [ ]:
# List of inverted tapers names.
tap_names = ["linear", "quadratic", "exponential"]


def get_taper(
    taper_shape="linear",
    init_coord=[pad_x, taper_w_in / 2],
    end_coord=[pad_x + taper_l, taper_w_out / 2],
):
    """Return a polygon composed of the taper vertices given the desired shape."""

    x0 = init_coord[0]
    x1 = end_coord[0]
    y0 = init_coord[1]
    y1 = end_coord[1]
    x = np.linspace(x0, x1, 41)

    tap_name = "linear_tap"
    if taper_shape == "quadratic":
        alpha = ((x - x0) / (x1 - x0)) ** 2
        tap_name = "quadratic_tap"
    elif taper_shape == "exponential":
        alpha = np.expm1((x - x0) / (x1 - x0)) / np.expm1(1)
        tap_name = "exponential_tap"
    elif taper_shape == "linear":
        alpha = (x - x0) / (x1 - x0)
    else:
        print(
            "taper_shape is neither 'linear', 'quadratic', or 'exponential'!\n"
            + "Linear taper shape returned!"
        )
        alpha = (x - x0) / (x1 - x0)

    y = y0 + alpha * (y1 - y0)
    upper_bound = [[xv, yv] for xv, yv in zip(x, y)] + [[_inf, y1]]
    lower_bound = [[_inf, -y1]] + [[xv, -yv] for xv, yv in zip(x[::-1], y[::-1])]
    taper_polygon = upper_bound + lower_bound

    # Inverted taper structure using a PolySlab.
    taper = td.Structure(
        geometry=td.PolySlab(
            vertices=taper_polygon, axis=2, slab_bounds=(-taper_t / 2, taper_t / 2)
        ),
        medium=mat_sin,
        name=tap_name,
    )
    return taper

In [ ]:
def get_fiber_structure(r_core, l_fiber, center):
    core_fiber = td.Structure(
    geometry=td.Cylinder(
        axis=0,
        radius=r_core,
        length=l_fiber,
        center=center,
    ),
    medium=mat_core_test,
    name="fiber_core"
    )

    return core_fiber

In [ ]:
def plot_simulation_data(sim_data, title: Optional[str] = None):
    mode_amps = sim_data["mode_0"]
    coeffs_f = mode_amps.amps.sel(direction="+")
    power_0 = np.abs(coeffs_f.sel(mode_index=0)) ** 2
    power_0_db = 10 * np.log10(power_0)

    # Extract power at central frequency using the objective's pattern
    monitor = sim_data["mode_0"]
    amps = monitor.amps.sel(mode_index=0, direction="+")
    amps_np = amps.values
    freq_idx = np.argmin(np.abs(amps.f.values - freq_c))
    power_c = np.abs(amps_np[freq_idx]) ** 2
    power_c_dB = 10 * np.log10(power_c)

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(
        wl_range,
        power_0_db,
        linestyle="solid",
        linewidth=2.0,
        color="#0F6E56",
        label="TE$_0$ coupling efficiency",
    )

    # Shade under the curve
    ax.fill_between(wl_range, power_0_db, float(power_0_db.min()) - 0.05,
                    alpha=0.08, color="#0F6E56")

    # Mark λ_c with a vertical dashed line
    ax.axvline(wl_c, color="gray", linewidth=0.8,
               linestyle="--", alpha=0.6)

    # Horizontal dashed line to y-axis at power_c_dB
    ax.axhline(power_c_dB, color="#D85A30", linewidth=0.8,
               linestyle=":", alpha=0.7)

    # Dot on the curve at λ_c
    ax.scatter([wl_c], [power_c_dB],
               color="#D85A30", zorder=5, s=60, label=f"{wl_c} µm → {power_c_dB:.3f} dB")

    # # Annotate the value
    # ax.annotate(
    #     f"{power_c_dB:.3f} dB",
    #     xy=(wl_range[0], power_c_dB),
    #     xytext=(wl_range[0] + 0.002, power_c_dB + 0.01),
    #     fontsize=9,
    #     color="#D85A30",
    #     ha="left",
    # )

    ax.set_xlim(wl_range[0], wl_range[-1])
    ax.set_xlabel("Wavelength (µm)", fontsize=12)
    ax.set_ylabel("Coupling Efficiency (dB)", fontsize=12)
    ax.tick_params(labelsize=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

    if title:
        ax.set_title(title, fontsize=13, fontweight="bold", pad=10)

    ax.legend(fontsize=10, framealpha=0.5)
    fig.tight_layout()
    plt.show()

In [ ]:
def get_simulations(
    tap_length=taper_l,
    tap_w_in=taper_w_in,
    tap_w_out=taper_w_out,
    tap_t=taper_t,
    box_thick=box_thick,
    clad_thick=clad_thick,
    fixed_length: bool = True,
    gap: float = gap,
    three_d: bool = True,
    fiber_mode_source: bool = True,
    source_x: float = fiber_length - gap - wl_c,
    fiber_length: float = fiber_length,
    full_domain: bool = False
):
    """Return a simulation object with the specified geometry.
    
    Args:
    tap_length: taper length.
    tap_w_in: taper width at input.
    tap_w_out: taper width at its end, equivalent to the specified waveguide width.
    tap_t: taper (and wabeguide) thickness.
    box_thich: oxide thickness below the waveguide.
    clad_thick: oxide thickness above the waveguide.
    fixed_length: adjoint optimization requires fixed simulation size. When True, the simulation size is defined by the maximum
        taper length under consideration, so the straight waveguide part extends to accommodate. If False, then the simulation
        size depends on the input taper length.
    gap: gap between the end of the fiber and the start of the chip
    three_d: when False, the y-dimension of the simulation is set to 0, slab waveguide. Only used for checks.
    fiber_mode_source: when True the source of the simulation is the eigenmode of the fiber model. Otherwise a Gaussian source is used.
    source_x: x-coordinate of source.
    fiber_length: length of fiber structure, indicating the start of the chip. In practice, the fiber ends at fiber_length -gap,
        and the chip starts at fiber_length.
    full_domain: when False the y-size of the simulation is a 3*wl extension from the waveguide to reduce computational cost. 
    """
    sim_size_x = fiber_length + taper_l_max + pad_x
    size_x = fiber_length + tap_length + pad_x
    if not fixed_length:
        sim_size_x = size_x
    size_y = 16 if full_domain else tap_w_out + 2 * pad_y
    size_z = 16 if full_domain else box_thick + clad_thick + tap_t + 2.0

    # ---------------------------------------------------------
    # Fiber core structure
    # ---------------------------------------------------------
    fiber_structure = get_fiber_structure(r_core=core_diameter/2, l_fiber=fiber_length + 2.0, center=(fiber_length/2 - gap - 1.0, 0, 0))

    fiber_cladding = td.Structure(
        geometry=td.Box.from_bounds(rmin=(-_inf, -_inf, -_inf), rmax=(fiber_length - gap, _inf, _inf)),
        medium=mat_sio2,
        name="fiber_clad"
    )

    # Silicon dioxide box + cladding layers
    sio2_box = td.Structure(
        geometry=td.Box.from_bounds(rmin=(fiber_length, -_inf, -box_thick), rmax=(_inf, _inf, 0)),
        medium=mat_sio2,
        name="box"
    )

    sio2_clad = td.Structure(
        geometry=td.Box.from_bounds(rmin=(fiber_length, -_inf, 0), rmax=(_inf, _inf, clad_thick)),
        medium=mat_sio2,
        name="clad"
    )

    # Silicon dioxide box + cladding layers
    si_substrate = td.Structure(
        geometry=td.Box.from_bounds(rmin=(fiber_length, -_inf, -_inf), rmax=(_inf, _inf, -box_thick)),
        medium=mat_si,
        name="substrate"
    )

    taper_poly = get_taper(
            taper_shape="linear",
            init_coord=[fiber_length, tap_w_in / 2],
            end_coord=[fiber_length + tap_length, tap_w_out / 2],
        )

    if fiber_mode_source:
        mode_spec = td.ModeSpec(num_modes=1, target_neff=n_core)
        fiber_mode = td.ModeSource(
            center=(source_x, 0, 0),
            size=(0, 16, 16),
            source_time=td.GaussianPulse(freq0=freq_c, fwidth=freq_width),
            direction="+",
            mode_spec=mode_spec,
            mode_index=0,
        )
        source = [fiber_mode]
        structures = [sio2_box, fiber_cladding, sio2_clad, taper_poly, fiber_structure, si_substrate]
    else:
        # Gaussian source focused at the taper tip.
        # The Gaussian source must be placed at a plane within a uniform medium.
        gauss_source = td.GaussianBeam(
            center=(source_x, 0, 0),
            size=(0, 16, 16),
            source_time=td.GaussianPulse(freq0=freq_c, fwidth=freq_width),
            pol_angle=0,
            direction="+",
            num_freqs=7,
            waist_radius=spot_size / 2,
            waist_distance=-(fiber_length - source_x)
        )
        source = [gauss_source]
        structures = [sio2_box, fiber_cladding, sio2_clad, taper_poly, si_substrate]

    # Field monitor to visualize the fields throughout the length of the taper.
    field_monitor_xy = td.FieldMonitor(
        center=(sim_size_x / 2, 0, 0),
        size=(sim_size_x, size_y, 0),
        freqs=[freq_c],
        name="field_xy",
    )

    # Field monitor to visualize the Gaussian input fields.
    field_input = td.FieldMonitor(
        center=(fiber_length, 0, 0),
        size=(0, size_y, size_z),
        freqs=[freq_c],
        name="field_input",
    )

    # Field monitor to visualize the fields at the output waveguide.
    field_output = td.FieldMonitor(
        center=(sim_size_x - pad_x / 2, 0, 0),
        size=(0, size_y, size_z),
        freqs=[freq_c],
        name="field_output",
    )

    # Mode monitor to get the power coupled into the fundamental TE mode.
    mode_spec = td.ModeSpec(num_modes=1)
    mode_monitor = td.ModeMonitor(
        center=(sim_size_x - pad_x / 2, 0, 0),
        size=(0, size_y, box_thick + clad_thick),
        freqs=freq_range,
        mode_spec=mode_spec,
        name="mode_0",
    )

    sim = td.Simulation(
        center=(sim_size_x / 2, 0, 0),
        size=(sim_size_x, size_y if three_d else 0, size_z),
        medium=mat_air,
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=20, wavelength=wl_c),
        structures=structures,
        sources=source,
        normalize_index=0,
        monitors=[field_monitor_xy, mode_monitor, field_input, field_output],
        boundary_spec=td.BoundarySpec(
            x=td.Boundary.pml(),
            y=td.Boundary.pml() if three_d else td.Boundary(
                plus=td.PMCBoundary(),   # PMC → TE-like (Ey polarization)
                minus=td.PMCBoundary(),  # use PECBoundary() for TM-like (Ez)
            ),
            z=td.Boundary.pml()
        ),
        symmetry=(0, -1 if three_d else 0, 1),
        run_time=run_time,
    )
    
    return sim

In [ ]:
sim = get_simulations(
        tap_w_in=taper_w_in,
        tap_length=taper_l,
        tap_w_out=taper_w_out,
        tap_t=taper_t,
        fixed_length=False,
        # fiber_mode_source=False
    )

In [ ]:
ax = sim.plot_eps(y=0)
ax.set_aspect('auto')
ax.set_xlim(7.8,8.2)

In [ ]:
sim.plot_3d()

In [ ]:
# sim_data = web.run(sim)
# plot_simulation_data(sim_data=sim_data)

# 2. Optimization through adjoint method

In [ ]:
def objective(params)->float:
    """Returns the negative of the power at the central wavelength. This is the metric which will be the optimization objective.
    Since we want to maximize power through gradient descent, we consider -power as a 'loss'.

    Args:
    params: optimization parameters.
    """
    sim = get_simulations(
        tap_w_in=params[0],
        tap_length=params[1],
    )
    
    result = web.run(sim, task_name="edge_coupler_adjoint", verbose=False)
    
    # Use the same pattern as get_mode_monitor_power — stay in autograd.numpy
    monitor = result["mode_0"]
    amps = monitor.amps.sel(mode_index=0, direction="+")
    
    # Convert to numpy FIRST, then use autograd.numpy operations
    amps_np = amps.values  # get underlying numpy array
    freq_idx = np.argmin(np.abs(amps.f.values - freq_c))  # find nearest freq index
    
    power = np.abs(amps_np[freq_idx]) ** 2  # autograd.numpy operations on numpy array
    
    return -power  # plain numpy scalar, autograd can trace this

In [ ]:
# Only using 2 optimization parameters, since other parameters like output width and thickness are fixed in the paper.
p_min = anp.array([taper_w_in_min, taper_l_min])
p_max = anp.array([taper_w_in_max, taper_l_max])

In [ ]:
def normalize(params_physical):
    """Map physical parameters into the [0,1] normalized space."""
    return (params_physical - p_min) / (p_max - p_min)

def denormalize(params_norm):
    """Map parameters from the [0,1] normalized space back into their physical dimensions"""
    return p_min + params_norm*(p_max - p_min)

def objective_normalized(params_norm) -> float:
    """Wrapper around objective function, so that it is ran through normalized parameters"""
    return objective(denormalize(params_norm))

In [ ]:
# Initial physical parameters
params_phys_init = anp.array([taper_w_in, taper_l])

# Normalized initial point
params_norm = normalize(params_phys_init)

# Learning rate
learning_rate = 0.01

optimizer = optax.adam(learning_rate=learning_rate, b1=0.8, b2=0.999)
opt_state = optimizer.init(params_norm)

# We basically have one input, which is a vector. So no need for argnum
obj_val_and_grad = value_and_grad(objective_normalized) 

In [ ]:
# ---------------------------------
# Checkpoint
# ---------------------------------

os.makedirs("misc", exist_ok=True)
restart_file_name = "misc/sin_edge_coupler_opt_state.pkl"

num_steps = 10

figure_of_merit = np.zeros(num_steps+1)
power_evolution = []

def pack_state(step_index):
    return {
        "iteration": step_index,
        "params_norm": params_norm,
        "opt_state": opt_state,
        "figure_of_merit": figure_of_merit,
        "power_evolution": power_evolution
    }

restart_optimization = os.path.exists(restart_file_name)
start_idx = 0

if restart_optimization:
    with open(restart_file_name, "rb") as fh:
        ckpt = pickle.load(fh)
    
    start_idx = ckpt["iteration"]
    params_norm = ckpt["params_norm"]
    opt_state = ckpt["opt_state"]
    power_evolution = ckpt["power_evolution"]

    # Extend figure_of_merit to the new num_steps if it grew
    old_fom = ckpt["figure_of_merit"]
    figure_of_merit = np.zeros(num_steps + 1)
    figure_of_merit[:len(old_fom)] = old_fom   # preserve existing values
    
    print(f"Resuming from iteration {start_idx}")
else:
    with open(restart_file_name, "wb") as fh:
        pickle.dump(pack_state(0), fh)

In [ ]:
restart_file_name = "misc/sin_edge_coupler_opt_state.pkl"

with open(restart_file_name, "rb") as fh:
    ckpt = pickle.load(fh)

params_norm_final = ckpt["params_norm"]
p_phys_final = denormalize(params_norm_final)

fab_resolution = 0.005
p_phys_final[0] = anp.round(p_phys_final[0] / fab_resolution) * fab_resolution

print(f"Iteration reached: {ckpt['iteration']}")
print(f"w_in   = {float(p_phys_final[0]):.4f} µm")
print(f"length = {float(p_phys_final[1]):.1f} µm")
print(f"FOM history: {ckpt['figure_of_merit']}")

In [ ]:
for step_idx in range(start_idx, num_steps):
    # Check before expensive simulation
    with open(restart_file_name, "wb") as fh:
        pickle.dump(pack_state(step_idx), fh)

    f, g = obj_val_and_grad(params_norm)
    figure_of_merit[step_idx] = f
    power_dB = 10*np.log10(-f)

    # periodic full-bandwidth evaluation

    # Adam update in normalized space
    updates, opt_state = optimizer.update(g, opt_state, params_norm)
    params_norm = optax.apply_updates(params_norm, updates)

    # Clip to [0,1] in normalized space to account for physical bound
    params_norm = anp.clip(params_norm, 0.0, 1.0)

    # Logging
    p_phys = denormalize(params_norm=params_norm)

    fab_resolution = 0.005  # 5 nm grid, in µm

    p_phys[0] = anp.round(p_phys[0] / fab_resolution) * fab_resolution

    print(
        f"[{step_idx+1:3d}/{num_steps}] "
        f"coupling = {power_dB:.2f} dB | "
        f"w_in = {float(p_phys[0]):.4f} µm | "
        f"length = {float(p_phys[1]):.1f} µm | "
    )

# Save final state after loop completes
with open(restart_file_name, "wb") as fh:
    pickle.dump(pack_state(num_steps), fh)

# 3. Final evaluation

## 3.1 View of space-restricted simulation spectrum

In [ ]:
sim = get_simulations(
        tap_w_in=p_phys[0],
        tap_length=p_phys[1],
        fixed_length=False,
    )

In [ ]:
sim_data = web.run(sim)

In [ ]:
fig, axs = plt.subplots(1, 1, tight_layout=True, figsize=(10, 2))

sim_data.plot_field("field_xy", "Ey", f=freq_c, val="abs", ax=axs)
axs.set_aspect("auto")  # Used to better visualize the shapes.
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, tight_layout=True, figsize=(8, 4))
sim_data.plot_field("field_input", "Ey", f=freq_c, val="abs", ax=ax1)
ax1.set_title("|Ey| at taper tip")
ax1.collections[-1].set_clim(0, 20)
ax1.set_xlim(-2, 2)
ax1.set_ylim(-2, 2)

sim_data.plot_field("field_output", "Ey", f=freq_c, val="abs", ax=ax2)
ax2.set_title("|Ey| at taper output")
ax2.collections[-1].set_clim(0, 50)
ax2.set_xlim(-2, 2)
ax2.set_ylim(-2, 2)
plt.show()

In [ ]:
plot_simulation_data(sim_data=sim_data, title="Edge coupler with w_in = 0.41 µm, length = 291.5 µm taper")

In [ ]:
sim_full = get_simulations(
        tap_w_in=p_phys[0],
        tap_length=p_phys[1],
        fixed_length=False,
        full_domain=True
    )

## 3.2 Full-domain simulation

In [ ]:
sim_full_data = web.run(sim_full, path="full_domain.hdf5")
# In case simulation parameters are different, but want to retrieve data from specfici file:
# sim_full_data = td.SimulationData.from_file("full_domain.hdf5")

In [ ]:
plot_simulation_data(sim_data=sim_full_data, title=f"Edge coupler with w_in = {p_phys[0]} µm, length = {round(p_phys[1],1)} µm taper")

In [ ]:
fig, axs = plt.subplots(1, 1, tight_layout=True, figsize=(10, 2))

sim_full_data.plot_field("field_xy", "Ey", f=freq_c, val="abs", ax=axs)
axs.set_aspect("auto")  # Used to better visualize the shapes.
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, tight_layout=True, figsize=(8, 4))
sim_full_data.plot_field("field_input", "Ey", f=freq_c, val="abs", ax=ax1)
ax1.set_title("|Ey| at taper tip")
ax1.collections[-1].set_clim(0, 20)
ax1.set_xlim(-2, 2)
ax1.set_ylim(-2, 2)

sim_full_data.plot_field("field_output", "Ey", f=freq_c, val="abs", ax=ax2)
ax2.set_title("|Ey| at taper output")
ax2.collections[-1].set_clim(0, 50)
ax2.set_xlim(-2, 2)
ax2.set_ylim(-2, 2)
plt.show()